# Approach 2: Anchor-Based AKI Risk Stratification — GPC-Aligned Features

**Feature collection:** admission → anchor − LOOKBACK_HOURS (AKI) / anchor (non-AKI)

**GPC alignment:** Lab and vital panels extended to match the GPC RF feature importance lists (KUMC, MCW, UIOWA, UPITT, UTSW, UofU). Added calcium, chloride, phosphate, magnesium, total protein, direct bilirubin, RDW, basophil%, lymphocyte%, and BMI — all present in the GPC `in_shared98` top-ranked feature set but previously missing from MIMIC-IV extraction. Feature groups (Step 15A) are organised to mirror GPC `feature_class` categories for consistent cross-dataset feature-sparsity simulation.

**Lead time:** `LOOKBACK_HOURS=24` (aligned with GPC pre-processing); `LOOKBACK_HOURS=48` available as secondary comparison.


## 1. Setup & Authentication

In [ ]:
!pip install google-cloud-bigquery pandas numpy matplotlib seaborn db-dtypes --quiet

In [ ]:
import pandas as pd
import numpy as np
from google.cloud import bigquery
from google.colab import auth
import os, warnings
warnings.filterwarnings('ignore')
auth.authenticate_user()
print('✓ Authentication successful!')


✓ Authentication successful!


In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────
PROJECT_ID              = 'mimic-aki-483914'           # ← UPDATE IF NEEDED
DATASET                 = 'physionet-data.mimiciv_3_1'

LOOKBACK_HOURS          = 24     # lead time: 48h primary (ADQI), 24h secondary

# Age restriction:
# Liu et al. 2018 used 18-64. Relaxed to 18+ here because:
#   (1) MIMIC-IV skews older than Liu et al. dataset
#   (2) AKI incidence rises with age — capping at 64 halves expected prevalence
#   (3) General ward AKI rate is 12-17%; age cap produces ~6% (too low)
AGE_MIN                 = 18
AGE_MAX                 = None   # None = no upper age limit

# SCr admission threshold (Liu et al. 2018):
# Exclude SCr > 1.3 mg/dL within 24h of admission
# (pre-existing kidney dysfunction — label ambiguity with KDIGO)
# Note: CKD ICD-code exclusion removed — SCr threshold handles this implicitly
# as in Liu et al. (who did not explicitly exclude CKD via diagnosis codes)
SCR_ADMISSION_THRESHOLD = 1.3    # mg/dL

OUTPUT_CSV = f'aki_anchor_based_{LOOKBACK_HOURS}h_lookback_aligned_features.csv'

client = bigquery.Client(project=PROJECT_ID)

print('='*70)
print('APPROACH 2: ANCHOR-BASED AKI RISK STRATIFICATION')
print('Aligned with Liu et al. 2018 (AMIA, PMC5977670)')
print('='*70)
print(f'  Lookback:        {LOOKBACK_HOURS}h')
print(f'  Age range:       {AGE_MIN}+ years (no upper cap)')
print(f'  SCr threshold:   >{SCR_ADMISSION_THRESHOLD} mg/dL DEFINED but exclusion DISABLED (see Step 5A) -- retaining these patients')
print(f'  CKD exclusion:   SCr threshold only (no ICD-code exclusion)')
print(f'  Output:          {OUTPUT_CSV}')
print('='*70)


APPROACH 2: ANCHOR-BASED AKI RISK STRATIFICATION — GPC-ALIGNED FEATURES
Aligned with Liu et al. 2018 (AMIA, PMC5977670)
  Lookback:        24h
  Age range:       18+ years (no upper cap)
  SCr threshold:   >1.3 mg/dL at admission -> excluded
  CKD exclusion:   SCr threshold only (no ICD-code exclusion)
  Output:          aki_anchor_based_24h_lookback_aligned_features.csv


## 2. Last Hospital Admission per Patient
*(Reused from Approach 1)*

In [ ]:
%%time
print('STEP 1: LAST HOSPITAL ADMISSION PER PATIENT')

query_last_admission = f"""
WITH ranked AS (
  SELECT subject_id, hadm_id, admittime, dischtime, admission_type, insurance,
         ROW_NUMBER() OVER (PARTITION BY subject_id ORDER BY admittime DESC) AS rn
  FROM `{DATASET}_hosp.admissions`
)
SELECT subject_id, hadm_id, admittime, dischtime, admission_type, insurance
FROM ranked WHERE rn = 1
ORDER BY subject_id
"""
df_last_encounters = client.query(query_last_admission).to_dataframe()
df_last_encounters['stay_hours'] = (
    (df_last_encounters['dischtime'] - df_last_encounters['admittime'])
    .dt.total_seconds() / 3600
)
# Minimum stay: at least LOOKBACK_HOURS so a feature window exists
before = len(df_last_encounters)
df_last_encounters = df_last_encounters[
    df_last_encounters['stay_hours'] >= LOOKBACK_HOURS
].copy()
print(f'  ✓ {before:,} total  →  {len(df_last_encounters):,} after >{LOOKBACK_HOURS}h stay filter')
df_last_encounters.head()


STEP 1: LAST HOSPITAL ADMISSION PER PATIENT
  ✓ 223,452 total  →  171,462 after >24h stay filter
CPU times: user 217 ms, sys: 90.5 ms, total: 307 ms
Wall time: 4.42 s


,subject_id,hadm_id,admittime,dischtime,admission_type,insurance,stay_hours
0,10000032,25742920,2180-08-05 23:44:00,2180-08-07 17:50:00,EW EMER.,Medicaid,42.100000
4,10000117,27988844,2183-09-18 18:10:00,2183-09-21 16:30:00,OBSERVATION ADMIT,Medicaid,70.333333
8,10000560,28979390,2189-10-15 10:30:00,2189-10-17 15:00:00,SURGICAL SAME DAY ADMISSION,Private,52.500000
10,10000690,26146595,2152-01-28 23:40:00,2152-01-30 15:56:00,EW EMER.,Medicare,40.266667
11,10000719,24558333,2140-04-15 00:14:00,2140-04-18 12:29:00,URGENT,Private,84.250000


## 3. Demographics
*(Reused from Approach 1)*

In [ ]:
%%time
print('STEP 2: DEMOGRAPHICS')

df_patients = client.query(
    f"SELECT subject_id, gender, anchor_age, dod FROM `{DATASET}_hosp.patients`"
).to_dataframe()
df_demographics = df_last_encounters.merge(df_patients, on='subject_id', how='left')
df_demographics['age_at_admission'] = df_demographics['anchor_age']
print(f'  ✓ {len(df_demographics):,} patients  |  mean age {df_demographics["age_at_admission"].mean():.1f}')
df_demographics.head()

STEP 2: DEMOGRAPHICS
  ✓ 171,462 patients  |  mean age 57.6
CPU times: user 171 ms, sys: 56.5 ms, total: 227 ms
Wall time: 3.03 s


,subject_id,hadm_id,admittime,dischtime,admission_type,insurance,stay_hours,gender,anchor_age,dod,age_at_admission
0,10000032,25742920,2180-08-05 23:44:00,2180-08-07 17:50:00,EW EMER.,Medicaid,42.100000,F,52,2180-09-09,52
1,10000117,27988844,2183-09-18 18:10:00,2183-09-21 16:30:00,OBSERVATION ADMIT,Medicaid,70.333333,F,48,NaT,48
2,10000560,28979390,2189-10-15 10:30:00,2189-10-17 15:00:00,SURGICAL SAME DAY ADMISSION,Private,52.500000,F,53,NaT,53
3,10000690,26146595,2152-01-28 23:40:00,2152-01-30 15:56:00,EW EMER.,Medicare,40.266667,F,86,2152-01-30,86
4,10000719,24558333,2140-04-15 00:14:00,2140-04-18 12:29:00,URGENT,Private,84.250000,F,34,NaT,34


## 3A. Age Restriction

Liu et al. 2018 restrict to **age 18–64** at admission.


In [ ]:
print('STEP 3A: AGE RESTRICTION')

before = len(df_demographics)
if AGE_MIN is not None:
    df_demographics = df_demographics[
        df_demographics['age_at_admission'] >= AGE_MIN
    ].copy()
if AGE_MAX is not None:
    df_demographics = df_demographics[
        df_demographics['age_at_admission'] <= AGE_MAX
    ].copy()
print(f'  Before: {before:,}  After: {len(df_demographics):,}  '
      f'Removed: {before - len(df_demographics):,}')
age_str = f'{AGE_MIN}+' if AGE_MAX is None else f'{AGE_MIN}-{AGE_MAX}'
print(f'  Age filter applied: {age_str}')
print(f'  Mean age: {df_demographics["age_at_admission"].mean():.1f} years')
print(f'  Age range: {df_demographics["age_at_admission"].min():.0f} - '
      f'{df_demographics["age_at_admission"].max():.0f}')


STEP 3A: AGE RESTRICTION
  Before: 171,462  After: 171,462  Removed: 0
  Age filter applied: 18+
  Mean age: 57.6 years
  Age range: 18 - 91


## 4. Comorbidities
*(Reused from Approach 1)*

In [ ]:
%%time
print('STEP 3: COMORBIDITIES')

hadm_ids = df_demographics['hadm_id'].tolist()
query_comorbidities = f"""
SELECT hadm_id,
    MAX(CASE WHEN icd_code LIKE 'E11%' OR icd_code LIKE '250%' THEN 1 ELSE 0 END) AS has_diabetes,
    MAX(CASE WHEN icd_code LIKE 'I10%' OR icd_code LIKE '401%' THEN 1 ELSE 0 END) AS has_hypertension,
    MAX(CASE WHEN icd_code LIKE 'I50%' OR icd_code LIKE '428%' THEN 1 ELSE 0 END) AS has_chf,
    MAX(CASE WHEN icd_code LIKE 'A41%' OR icd_code LIKE '038%' THEN 1 ELSE 0 END) AS has_sepsis,
    MAX(CASE WHEN icd_code LIKE 'K70%' OR icd_code LIKE 'K74%' OR icd_code LIKE '571%' THEN 1 ELSE 0 END) AS has_liver_disease,
    MAX(CASE WHEN icd_code LIKE 'C%'   OR (icd_code >= '140' AND icd_code < '210')  THEN 1 ELSE 0 END) AS has_cancer,
    MAX(CASE WHEN icd_code LIKE 'N18%' OR icd_code LIKE '585%' THEN 1 ELSE 0 END) AS has_ckd
FROM `{DATASET}_hosp.diagnoses_icd`
WHERE hadm_id IN UNNEST(@hadm_ids)
GROUP BY hadm_id
"""
job_config = bigquery.QueryJobConfig(
    query_parameters=[bigquery.ArrayQueryParameter('hadm_ids', 'INT64', hadm_ids)]
)
df_comorbidities = client.query(query_comorbidities, job_config=job_config).to_dataframe()
df_demographics  = df_demographics.merge(df_comorbidities, on='hadm_id', how='left')
for col in ['has_diabetes','has_hypertension','has_chf','has_sepsis',
            'has_liver_disease','has_cancer','has_ckd']:
    df_demographics[col] = df_demographics[col].fillna(0).astype(int)
print(f'  ✓ Comorbidities merged for {len(df_demographics):,} patients')

STEP 3: COMORBIDITIES
  ✓ Comorbidities merged for 171,462 patients
CPU times: user 1.21 s, sys: 159 ms, total: 1.37 s
Wall time: 18.1 s


## 5. All Creatinine Measurements
*(Reused from Approach 1)*

In [ ]:
%%time
print('STEP 4: ALL CREATININE MEASUREMENTS')

CACHE_SCR = 'cached_creatinine_approach2.csv'
subject_ids      = df_demographics['subject_id'].tolist()
earliest_admit   = df_demographics['admittime'].min()
latest_discharge = df_demographics['dischtime'].max()
lookback_date    = earliest_admit - pd.Timedelta(days=365)

if os.path.exists(CACHE_SCR):
    print('  ✓ Loading from cache...')
    df_scr_all = pd.read_csv(CACHE_SCR, parse_dates=['charttime'])
else:
    query_scr = f"""
    SELECT le.subject_id, le.hadm_id, le.charttime, le.valuenum AS creatinine_mg_dl
    FROM `{DATASET}_hosp.labevents` le
    WHERE le.subject_id IN UNNEST(@subject_ids)
        AND le.itemid = 50912
        AND le.valuenum IS NOT NULL AND le.valuenum > 0 AND le.valuenum < 20
        AND DATE(le.charttime) >= DATE(@lookback_date)
        AND DATE(le.charttime) <= DATE(@latest_discharge)
        AND le.charttime >= @lookback_date AND le.charttime <= @latest_discharge
    ORDER BY le.subject_id, le.charttime
    """
    job_config = bigquery.QueryJobConfig(query_parameters=[
        bigquery.ArrayQueryParameter('subject_ids',    'INT64',    subject_ids),
        bigquery.ScalarQueryParameter('lookback_date', 'DATETIME', lookback_date),
        bigquery.ScalarQueryParameter('latest_discharge','DATETIME', latest_discharge),
    ])
    df_scr_all = client.query(query_scr, job_config=job_config).to_dataframe()
    df_scr_all.to_csv(CACHE_SCR, index=False)

print(f'  ✓ {len(df_scr_all):,} creatinine measurements  |  {df_scr_all["subject_id"].nunique():,} patients')

STEP 4: ALL CREATININE MEASUREMENTS
  ✓ 3,659,210 creatinine measurements  |  168,255 patients
CPU times: user 15.4 s, sys: 571 ms, total: 16 s
Wall time: 33.7 s


## 5A. SCr Admission Exclusion

Liu et al. 2018 exclude patients with SCr > 1.3 mg/dL within 24h of admission
(pre-existing kidney dysfunction — hospital-acquired AKI cannot be reliably distinguished).


In [ ]:
%%time
# ── STEP 5A: SCr ADMISSION EXCLUSION ── DISABLED ──────────────────────────
# Liu et al. 2018 exclude SCr > 1.3 mg/dL within 24h of admission
# COMMENTED OUT: retaining these patients to increase AKI prevalence
# and better represent real-world ward population including mild CKD

# before = len(df_demographics)
# df_demographics = df_demographics[
#     ~df_demographics['hadm_id'].isin(abnormal_scr)
# ].copy()
# print(f'  Excluded (SCr > {SCR_ADMISSION_THRESHOLD} mg/dL at admission): '
#       f'{before - len(df_demographics):,}')

print(f'  SCr admission exclusion SKIPPED — retaining all {len(df_demographics):,} patients')


  SCr admission exclusion SKIPPED — retaining all 171,462 patients
CPU times: user 56 µs, sys: 5 µs, total: 61 µs
Wall time: 64.1 µs


## 5B. GPC Shared98 Diagnosis Codes

Extracts the 70 ICD-9 diagnosis category codes present in GPC's real shared98
feature specification (confirmed directly against the 6-site GPC feature
list export), as individual binary flags per patient -- matching shared98's
structure exactly (70 separate fields, not aggregated into fewer summary
flags like the existing 6 comorbidity indicators above).

**ICD-9 only, by construction** (GPC's shared98 DX list contains no ICD-10
codes) -- see the limitation note in the code cell below.

In [ ]:
%%time
print('STEP: EXTRACT GPC SHARED98 DIAGNOSIS CODES')

# All 70 GPC shared98 DX codes are ICD-9 3-digit category codes (confirmed
# directly against the real GPC per-site feature lists -- no ICD-10 codes
# present in shared98's DX category at all).
#
# IMPORTANT LIMITATION, not yet resolved: MIMIC-IV contains a mix of ICD-9
# and ICD-10 coded admissions (ICD-10 became mandatory in the US in Oct
# 2015; MIMIC-IV spans well past that date). This extraction currently
# matches ICD-9 codes ONLY, following GPC's codes exactly as given.
# ICD-10-coded admissions will show 0 for all 70 flags below even if the
# patient genuinely has the condition -- a real, systematic gap, not
# resolved here. Fixing this properly needs a verified ICD-9-to-ICD-10 GEM
# crosswalk for these specific 70 codes (not attempted here to avoid
# introducing incorrect mappings from memory -- some codes like V58, 996
# do not have obvious 1:1 ICD-10 equivalents).

GPC_SHARED98_DX_CODES = ['041', '244', '250', '263', '272', '275', '276', '278', '280', '285', '287', '288', '296', '300', '305', '311', '327', '338', '357', '401', '403', '414', '416', '424', '425', '426', '427', '428', '429', '458', '491', '493', '496', '518', '530', '564', '571', '584', '585', '593', '599', '715', '719', '724', '729', '733', '780', '781', '782', '784', '785', '786', '787', '788', '789', '790', '793', '799', '995', '996', 'V05', 'V10', 'V12', 'V15', 'V43', 'V45', 'V49', 'V58', 'V72', 'V85']
assert len(GPC_SHARED98_DX_CODES) == 70

hadm_ids_dx = df_demographics['hadm_id'].tolist()

query_gpc_dx = f"""
SELECT hadm_id,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '041%' THEN 1 ELSE 0 END) AS dx_041,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '244%' THEN 1 ELSE 0 END) AS dx_244,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '250%' THEN 1 ELSE 0 END) AS dx_250,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '263%' THEN 1 ELSE 0 END) AS dx_263,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '272%' THEN 1 ELSE 0 END) AS dx_272,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '275%' THEN 1 ELSE 0 END) AS dx_275,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '276%' THEN 1 ELSE 0 END) AS dx_276,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '278%' THEN 1 ELSE 0 END) AS dx_278,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '280%' THEN 1 ELSE 0 END) AS dx_280,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '285%' THEN 1 ELSE 0 END) AS dx_285,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '287%' THEN 1 ELSE 0 END) AS dx_287,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '288%' THEN 1 ELSE 0 END) AS dx_288,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '296%' THEN 1 ELSE 0 END) AS dx_296,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '300%' THEN 1 ELSE 0 END) AS dx_300,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '305%' THEN 1 ELSE 0 END) AS dx_305,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '311%' THEN 1 ELSE 0 END) AS dx_311,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '327%' THEN 1 ELSE 0 END) AS dx_327,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '338%' THEN 1 ELSE 0 END) AS dx_338,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '357%' THEN 1 ELSE 0 END) AS dx_357,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '401%' THEN 1 ELSE 0 END) AS dx_401,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '403%' THEN 1 ELSE 0 END) AS dx_403,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '414%' THEN 1 ELSE 0 END) AS dx_414,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '416%' THEN 1 ELSE 0 END) AS dx_416,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '424%' THEN 1 ELSE 0 END) AS dx_424,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '425%' THEN 1 ELSE 0 END) AS dx_425,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '426%' THEN 1 ELSE 0 END) AS dx_426,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '427%' THEN 1 ELSE 0 END) AS dx_427,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '428%' THEN 1 ELSE 0 END) AS dx_428,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '429%' THEN 1 ELSE 0 END) AS dx_429,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '458%' THEN 1 ELSE 0 END) AS dx_458,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '491%' THEN 1 ELSE 0 END) AS dx_491,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '493%' THEN 1 ELSE 0 END) AS dx_493,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '496%' THEN 1 ELSE 0 END) AS dx_496,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '518%' THEN 1 ELSE 0 END) AS dx_518,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '530%' THEN 1 ELSE 0 END) AS dx_530,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '564%' THEN 1 ELSE 0 END) AS dx_564,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '571%' THEN 1 ELSE 0 END) AS dx_571,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '584%' THEN 1 ELSE 0 END) AS dx_584,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '585%' THEN 1 ELSE 0 END) AS dx_585,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '593%' THEN 1 ELSE 0 END) AS dx_593,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '599%' THEN 1 ELSE 0 END) AS dx_599,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '715%' THEN 1 ELSE 0 END) AS dx_715,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '719%' THEN 1 ELSE 0 END) AS dx_719,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '724%' THEN 1 ELSE 0 END) AS dx_724,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '729%' THEN 1 ELSE 0 END) AS dx_729,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '733%' THEN 1 ELSE 0 END) AS dx_733,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '780%' THEN 1 ELSE 0 END) AS dx_780,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '781%' THEN 1 ELSE 0 END) AS dx_781,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '782%' THEN 1 ELSE 0 END) AS dx_782,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '784%' THEN 1 ELSE 0 END) AS dx_784,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '785%' THEN 1 ELSE 0 END) AS dx_785,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '786%' THEN 1 ELSE 0 END) AS dx_786,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '787%' THEN 1 ELSE 0 END) AS dx_787,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '788%' THEN 1 ELSE 0 END) AS dx_788,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '789%' THEN 1 ELSE 0 END) AS dx_789,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '790%' THEN 1 ELSE 0 END) AS dx_790,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '793%' THEN 1 ELSE 0 END) AS dx_793,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '799%' THEN 1 ELSE 0 END) AS dx_799,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '995%' THEN 1 ELSE 0 END) AS dx_995,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '996%' THEN 1 ELSE 0 END) AS dx_996,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE 'V05%' THEN 1 ELSE 0 END) AS dx_V05,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE 'V10%' THEN 1 ELSE 0 END) AS dx_V10,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE 'V12%' THEN 1 ELSE 0 END) AS dx_V12,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE 'V15%' THEN 1 ELSE 0 END) AS dx_V15,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE 'V43%' THEN 1 ELSE 0 END) AS dx_V43,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE 'V45%' THEN 1 ELSE 0 END) AS dx_V45,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE 'V49%' THEN 1 ELSE 0 END) AS dx_V49,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE 'V58%' THEN 1 ELSE 0 END) AS dx_V58,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE 'V72%' THEN 1 ELSE 0 END) AS dx_V72,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE 'V85%' THEN 1 ELSE 0 END) AS dx_V85
FROM `{DATASET}_hosp.diagnoses_icd`
WHERE hadm_id IN UNNEST(@hadm_ids)
GROUP BY hadm_id
"""
job_config = bigquery.QueryJobConfig(
    query_parameters=[bigquery.ArrayQueryParameter('hadm_ids', 'INT64', hadm_ids_dx)]
)
df_gpc_dx = client.query(query_gpc_dx, job_config=job_config).to_dataframe()
df_demographics = df_demographics.merge(df_gpc_dx, on='hadm_id', how='left')

dx_cols = [f'dx_{c}' for c in GPC_SHARED98_DX_CODES]
for col in dx_cols:
    df_demographics[col] = df_demographics[col].fillna(0).astype(int)

print(f'  \u2713 {len(dx_cols)} GPC shared98 DX flags merged for {len(df_demographics):,} patients')
print(f'  \u26a0\ufe0f  ICD-9 only -- ICD-10-coded admissions undercounted, see note above')
print(f'  Prevalence of each flag (top 10 most common):')
print(df_demographics[dx_cols].mean().sort_values(ascending=False).head(10))


## 5C. Site-Specific (Non-Shared98) GPC Diagnosis Codes

Extracts the union of each real GPC site's OWN site-specific ICD-9 diagnosis
codes (beyond the universal 70 shared98 codes from Section 5B) -- genuine
inter-site heterogeneity from the real GPC network, mapped to the 5
simulated sites: `site_A`~UTSW, `site_B`~UPITT, `site_C`~MCW, `site_D`~KUMC,
`site_E`~UofU (same mapping as the GPC-realistic prevalence work).

Pooled here as one 86-code union since patients aren't yet split by site;
the FL simulation script's feature-group masking determines which of these
each simulated site actually sees, matching each site's real GPC counterpart.

In [ ]:
%%time
print('STEP: EXTRACT SITE-SPECIFIC (NON-SHARED98) GPC DIAGNOSIS CODES')

# Union of site-specific ICD-9 DX codes across the 5 real GPC sites mapped
# to the 5 simulated sites (site_A~UTSW, site_B~UPITT, site_C~MCW,
# site_D~KUMC, site_E~UofU -- same mapping used for the GPC-realistic
# prevalence work). These are each site's OWN non-shared98 diagnosis
# codes -- real, confirmed inter-site heterogeneity, not the universal 70.
#
# Extracted here as ONE pooled union (86 codes) since the cohort file is
# still patient-level, not yet split by simulated site. Per-site masking
# (which of these 86 columns each simulated site actually sees) happens
# in the FL simulation script, same mechanism as the existing feature
# groups (renal/metabolic_panel/etc.).

SITE_SPECIFIC_DX_CODES = ['038', '112', '197', '198', '238', '268', '274', '277', '279', '284', '289', '309', '348', '362', '366', '402', '404', '412', '433', '434', '437', '440', '443', '453', '459', '477', '478', '486', '492', '511', '514', '515', '516', '519', '536', '553', '562', '568', '569', '572', '573', '574', '577', '578', '592', '596', '600', '682', '707', '721', '722', '728', '783', '791', '792', '794', '796', '998', 'E03', 'E84', 'E87', 'E88', 'E93', 'E94', 'V01', 'V13', 'V14', 'V16', 'V17', 'V42', 'V44', 'V46', 'V53', 'V54', 'V59', 'V64', 'V65', 'V66', 'V67', 'V68', 'V70', 'V71', 'V73', 'V76', 'V87', 'V88']
assert len(SITE_SPECIFIC_DX_CODES) == 86

hadm_ids_dx2 = df_demographics['hadm_id'].tolist()

query_site_dx = f"""
SELECT hadm_id,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '038%' THEN 1 ELSE 0 END) AS dx_site_038,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '112%' THEN 1 ELSE 0 END) AS dx_site_112,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '197%' THEN 1 ELSE 0 END) AS dx_site_197,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '198%' THEN 1 ELSE 0 END) AS dx_site_198,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '238%' THEN 1 ELSE 0 END) AS dx_site_238,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '268%' THEN 1 ELSE 0 END) AS dx_site_268,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '274%' THEN 1 ELSE 0 END) AS dx_site_274,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '277%' THEN 1 ELSE 0 END) AS dx_site_277,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '279%' THEN 1 ELSE 0 END) AS dx_site_279,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '284%' THEN 1 ELSE 0 END) AS dx_site_284,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '289%' THEN 1 ELSE 0 END) AS dx_site_289,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '309%' THEN 1 ELSE 0 END) AS dx_site_309,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '348%' THEN 1 ELSE 0 END) AS dx_site_348,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '362%' THEN 1 ELSE 0 END) AS dx_site_362,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '366%' THEN 1 ELSE 0 END) AS dx_site_366,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '402%' THEN 1 ELSE 0 END) AS dx_site_402,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '404%' THEN 1 ELSE 0 END) AS dx_site_404,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '412%' THEN 1 ELSE 0 END) AS dx_site_412,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '433%' THEN 1 ELSE 0 END) AS dx_site_433,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '434%' THEN 1 ELSE 0 END) AS dx_site_434,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '437%' THEN 1 ELSE 0 END) AS dx_site_437,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '440%' THEN 1 ELSE 0 END) AS dx_site_440,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '443%' THEN 1 ELSE 0 END) AS dx_site_443,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '453%' THEN 1 ELSE 0 END) AS dx_site_453,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '459%' THEN 1 ELSE 0 END) AS dx_site_459,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '477%' THEN 1 ELSE 0 END) AS dx_site_477,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '478%' THEN 1 ELSE 0 END) AS dx_site_478,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '486%' THEN 1 ELSE 0 END) AS dx_site_486,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '492%' THEN 1 ELSE 0 END) AS dx_site_492,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '511%' THEN 1 ELSE 0 END) AS dx_site_511,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '514%' THEN 1 ELSE 0 END) AS dx_site_514,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '515%' THEN 1 ELSE 0 END) AS dx_site_515,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '516%' THEN 1 ELSE 0 END) AS dx_site_516,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '519%' THEN 1 ELSE 0 END) AS dx_site_519,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '536%' THEN 1 ELSE 0 END) AS dx_site_536,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '553%' THEN 1 ELSE 0 END) AS dx_site_553,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '562%' THEN 1 ELSE 0 END) AS dx_site_562,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '568%' THEN 1 ELSE 0 END) AS dx_site_568,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '569%' THEN 1 ELSE 0 END) AS dx_site_569,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '572%' THEN 1 ELSE 0 END) AS dx_site_572,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '573%' THEN 1 ELSE 0 END) AS dx_site_573,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '574%' THEN 1 ELSE 0 END) AS dx_site_574,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '577%' THEN 1 ELSE 0 END) AS dx_site_577,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '578%' THEN 1 ELSE 0 END) AS dx_site_578,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '592%' THEN 1 ELSE 0 END) AS dx_site_592,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '596%' THEN 1 ELSE 0 END) AS dx_site_596,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '600%' THEN 1 ELSE 0 END) AS dx_site_600,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '682%' THEN 1 ELSE 0 END) AS dx_site_682,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '707%' THEN 1 ELSE 0 END) AS dx_site_707,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '721%' THEN 1 ELSE 0 END) AS dx_site_721,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '722%' THEN 1 ELSE 0 END) AS dx_site_722,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '728%' THEN 1 ELSE 0 END) AS dx_site_728,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '783%' THEN 1 ELSE 0 END) AS dx_site_783,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '791%' THEN 1 ELSE 0 END) AS dx_site_791,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '792%' THEN 1 ELSE 0 END) AS dx_site_792,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '794%' THEN 1 ELSE 0 END) AS dx_site_794,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '796%' THEN 1 ELSE 0 END) AS dx_site_796,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE '998%' THEN 1 ELSE 0 END) AS dx_site_998,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE 'E03%' THEN 1 ELSE 0 END) AS dx_site_E03,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE 'E84%' THEN 1 ELSE 0 END) AS dx_site_E84,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE 'E87%' THEN 1 ELSE 0 END) AS dx_site_E87,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE 'E88%' THEN 1 ELSE 0 END) AS dx_site_E88,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE 'E93%' THEN 1 ELSE 0 END) AS dx_site_E93,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE 'E94%' THEN 1 ELSE 0 END) AS dx_site_E94,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE 'V01%' THEN 1 ELSE 0 END) AS dx_site_V01,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE 'V13%' THEN 1 ELSE 0 END) AS dx_site_V13,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE 'V14%' THEN 1 ELSE 0 END) AS dx_site_V14,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE 'V16%' THEN 1 ELSE 0 END) AS dx_site_V16,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE 'V17%' THEN 1 ELSE 0 END) AS dx_site_V17,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE 'V42%' THEN 1 ELSE 0 END) AS dx_site_V42,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE 'V44%' THEN 1 ELSE 0 END) AS dx_site_V44,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE 'V46%' THEN 1 ELSE 0 END) AS dx_site_V46,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE 'V53%' THEN 1 ELSE 0 END) AS dx_site_V53,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE 'V54%' THEN 1 ELSE 0 END) AS dx_site_V54,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE 'V59%' THEN 1 ELSE 0 END) AS dx_site_V59,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE 'V64%' THEN 1 ELSE 0 END) AS dx_site_V64,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE 'V65%' THEN 1 ELSE 0 END) AS dx_site_V65,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE 'V66%' THEN 1 ELSE 0 END) AS dx_site_V66,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE 'V67%' THEN 1 ELSE 0 END) AS dx_site_V67,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE 'V68%' THEN 1 ELSE 0 END) AS dx_site_V68,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE 'V70%' THEN 1 ELSE 0 END) AS dx_site_V70,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE 'V71%' THEN 1 ELSE 0 END) AS dx_site_V71,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE 'V73%' THEN 1 ELSE 0 END) AS dx_site_V73,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE 'V76%' THEN 1 ELSE 0 END) AS dx_site_V76,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE 'V87%' THEN 1 ELSE 0 END) AS dx_site_V87,
    MAX(CASE WHEN icd_version = 9 AND icd_code LIKE 'V88%' THEN 1 ELSE 0 END) AS dx_site_V88
FROM `{DATASET}_hosp.diagnoses_icd`
WHERE hadm_id IN UNNEST(@hadm_ids)
GROUP BY hadm_id
"""
job_config = bigquery.QueryJobConfig(
    query_parameters=[bigquery.ArrayQueryParameter('hadm_ids', 'INT64', hadm_ids_dx2)]
)
df_site_dx = client.query(query_site_dx, job_config=job_config).to_dataframe()
df_demographics = df_demographics.merge(df_site_dx, on='hadm_id', how='left')

dx_site_cols = [f'dx_site_{c}' for c in SITE_SPECIFIC_DX_CODES]
for col in dx_site_cols:
    df_demographics[col] = df_demographics[col].fillna(0).astype(int)

print(f'  \u2713 {len(dx_site_cols)} site-specific DX flags merged for {len(df_demographics):,} patients')
print(f'  Per-site code counts (for FL simulation masking config):')
print(f'    site_A (~UTSW):  67 codes')
print(f'    site_B (~UPITT): 24 codes')
print(f'    site_C (~MCW):   56 codes')
print(f'    site_D (~KUMC):  32 codes')
print(f'    site_E (~UofU):  31 codes')


## 6. Identify CKD Patients
*(Reused from Approach 1)*

In [ ]:
# CKD patients identified for reference only — NOT excluded
# Liu et al. 2018 did not explicitly exclude CKD via ICD codes.
# The SCr > 1.3 mg/dL admission threshold (Step 5A) implicitly handles
# most CKD patients. ICD-based exclusion was removed to:
#   (1) align with Liu et al.
#   (2) avoid deflating AKI prevalence below the expected 12-17% general ward rate
ckd_patients_set = set(
    df_demographics[df_demographics['has_ckd'] == 1]['subject_id']
)
print(f'  CKD patients (informational only, not excluded): {len(ckd_patients_set):,}  '
      f'({len(ckd_patients_set)/len(df_demographics)*100:.1f}%)')


  CKD patients (informational only, not excluded): 22,735  (13.3%)


## 7. Calculate Baseline SCr
*(Reused from Approach 1)*

In [ ]:
%%time
print('STEP 5: BASELINE SCr')

def calculate_mdrd_baseline_scr(age, sex, race='WHITE', target_egfr=75):
    sex_factor  = 0.742 if sex == 'F' else 1.0
    race_factor = 1.212 if race in ['BLACK', 'BLACK/AFRICAN AMERICAN'] else 1.0
    return (target_egfr / (175 * (age ** -0.203) * sex_factor * race_factor)) ** (-1/1.154)

def calculate_baseline_scr(row, df_scr_all):
    sid       = row['subject_id']
    admittime = row['admittime']
    patient_scr = df_scr_all[df_scr_all['subject_id'] == sid].sort_values('charttime')

    scr_first_24h_min = patient_scr[
        (patient_scr['charttime'] >= admittime) &
        (patient_scr['charttime'] <= admittime + pd.Timedelta(hours=24))
    ]['creatinine_mg_dl'].min()
    scr_first_24h_min = None if pd.isna(scr_first_24h_min) else scr_first_24h_min

    scr_7d     = patient_scr[
        (patient_scr['charttime'] >= admittime - pd.Timedelta(days=7)) &
        (patient_scr['charttime'] < admittime)
    ]['creatinine_mg_dl']
    scr_365to7 = patient_scr[
        (patient_scr['charttime'] >= admittime - pd.Timedelta(days=365)) &
        (patient_scr['charttime'] < admittime - pd.Timedelta(days=7))
    ]['creatinine_mg_dl']

    if len(scr_7d) > 0:
        ref = scr_7d.iloc[-1]
        baseline = min(ref, scr_first_24h_min) if scr_first_24h_min else ref
        method   = 'most_recent_7d'
    elif len(scr_365to7) > 0:
        ref = scr_365to7.mean()
        baseline = min(ref, scr_first_24h_min) if scr_first_24h_min else ref
        method   = 'avg_365to7'
    else:
        # No pre-admission SCr — use MDRD estimate for all patients
        # (previously CKD patients without history were excluded here;
        #  now we keep them and use MDRD, consistent with Liu et al.)
        mdrd = calculate_mdrd_baseline_scr(row['age_at_admission'], row['gender'])
        baseline = min(mdrd, scr_first_24h_min) if scr_first_24h_min else mdrd
        method   = 'mdrd'

    return pd.Series({'baseline_scr': baseline, 'baseline_method': method})

baseline_results = df_demographics.apply(
    lambda r: calculate_baseline_scr(r, df_scr_all), axis=1
)
df_demographics = pd.concat([df_demographics, baseline_results], axis=1)

# Drop patients with no baseline SCr (no SCr measurements at all)
before = len(df_demographics)
df_demographics = df_demographics[df_demographics['baseline_scr'].notna()].copy()
print(f'  Excluded (no SCr at all): {before - len(df_demographics):,}')
print(f'  Remaining: {len(df_demographics):,}')
print(df_demographics['baseline_method'].value_counts())


STEP 5: BASELINE SCr
  Excluded (no SCr at all): 0
  Remaining: 171,462
baseline_method
most_recent_7d    121753
mdrd               25349
avg_365to7         24360
Name: count, dtype: int64
CPU times: user 23min 8s, sys: 1.23 s, total: 23min 9s
Wall time: 23min 9s


## 8. Extract All Labs During Admission

In [ ]:
%%time
print('STEP 6: ALL LABS DURING ADMISSION')

# Lab panel aligned with GPC shared98 feature list (Zijian, KU shared drive).
# Added: calcium, chloride, phosphate, magnesium, RDW, basophils%, lymphocyte%,
# total_protein — all present in GPC top-ranked shared features but previously
# missing from MIMIC-IV extraction.
LAB_ITEMIDS = {
    'creatinine':     [50912],
    'bun':            [51006],
    'lactate':        [50813],
    'sodium':         [50983],
    'potassium':      [50971],
    'chloride':       [50902],            # GPC shared98 rank ~6
    'bicarbonate':    [50882],
    'calcium':        [50893],            # GPC shared98 rank ~8
    'phosphate':      [50970],            # GPC shared98 rank ~10
    'magnesium':      [50960],            # GPC shared98 (lower rank)
    'wbc':            [51301],
    'hemoglobin':     [51222, 50811],
    'platelets':      [51265, 51704],
    'glucose':        [50931, 50809],
    'albumin':        [50862],
    'total_protein':  [50976],            # GPC shared98 'Protein Mass'
    'bilirubin':      [50885],            # total bilirubin
    'bilirubin_dir':  [50883],            # direct bilirubin — GPC shared98
    'rdw':            [51277],            # RBC distribution width — GPC top feature
    'basophils_pct':  [51146],            # GPC shared98 'BasophilsPercent'
    'lymphocyte_pct': [51244],            # GPC top feature 'Lymphocyte Percent'
}
all_lab_itemids = [iid for ids in LAB_ITEMIDS.values() for iid in ids]
itemid_to_lab   = {iid: name for name, ids in LAB_ITEMIDS.items() for iid in ids}

CACHE_LABS = 'cached_labs_approach2_aligned.csv'  # renamed: new lab panel, avoid stale cache
hadm_ids   = df_demographics['hadm_id'].tolist()

if os.path.exists(CACHE_LABS):
    print('  ✓ Loading labs from cache...')
    df_labs_all = pd.read_csv(CACHE_LABS, parse_dates=['charttime'])
else:
    query_labs = f"""
    WITH adm AS (
        SELECT subject_id, hadm_id, admittime, dischtime
        FROM `{DATASET}_hosp.admissions`
        WHERE hadm_id IN UNNEST(@hadm_ids)
    )
    SELECT le.subject_id, le.hadm_id, le.itemid, le.charttime, le.valuenum
    FROM adm
    JOIN `{DATASET}_hosp.labevents` le
        ON adm.subject_id = le.subject_id
        AND le.charttime BETWEEN adm.admittime AND adm.dischtime
        AND le.itemid IN UNNEST(@itemids)
        AND le.valuenum IS NOT NULL AND le.valuenum > 0
        AND DATE(le.charttime) BETWEEN
            DATE_SUB(DATE(adm.admittime), INTERVAL 1 DAY)
            AND DATE_ADD(DATE(adm.dischtime), INTERVAL 1 DAY)
    ORDER BY le.subject_id, le.charttime
    """
    job_config = bigquery.QueryJobConfig(query_parameters=[
        bigquery.ArrayQueryParameter('hadm_ids', 'INT64', hadm_ids),
        bigquery.ArrayQueryParameter('itemids',  'INT64', all_lab_itemids),
    ])
    df_labs_all = client.query(query_labs, job_config=job_config).to_dataframe()
    df_labs_all.to_csv(CACHE_LABS, index=False)

df_labs_all['feature_name'] = df_labs_all['itemid'].map(itemid_to_lab)
print(f'  ✓ {len(df_labs_all):,} lab measurements  |  {df_labs_all["subject_id"].nunique():,} patients')

STEP 6: ALL LABS DURING ADMISSION
  ✓ 21 lab types queried (12 original + 9 GPC-aligned additions:
     calcium, chloride, phosphate, magnesium, total_protein,
     bilirubin_dir, rdw, basophils_pct, lymphocyte_pct)
  ✓ 13,940,852 lab measurements  |  159,318 patients
CPU times: user 68 s, sys: 2.41 s, total: 70.4 s
Wall time: 1min 38s


## 9. Extract All Vitals During Admission

In [ ]:
%%time
print('STEP 7: ALL VITALS DURING ADMISSION')

# Vitals panel aligned with GPC shared98 feature list.
# Added: bmi — present in GPC top-ranked shared VITAL_TIME features.
VITAL_ITEMIDS = {
    'heart_rate':  [220045],
    'sbp':         [220179],
    'dbp':         [220180],
    'resp_rate':   [220210],
    'spo2':        [220277],
    'temperature': [223761, 223762],
    'gcs_total':   [220739],
    'bmi':         [226512],          # GPC shared98 'ORIGINAL_BMI'
}
all_vital_itemids = [iid for ids in VITAL_ITEMIDS.values() for iid in ids]
itemid_to_vital   = {iid: name for name, ids in VITAL_ITEMIDS.items() for iid in ids}

CACHE_VITALS = 'cached_vitals_approach2_aligned.csv'  # renamed: new vital panel, avoid stale cache

if os.path.exists(CACHE_VITALS):
    print('  ✓ Loading vitals from cache...')
    df_vitals_all = pd.read_csv(CACHE_VITALS, parse_dates=['charttime'])
else:
    query_vitals = f"""
    WITH adm AS (
        SELECT subject_id, hadm_id, admittime, dischtime
        FROM `{DATASET}_hosp.admissions`
        WHERE hadm_id IN UNNEST(@hadm_ids)
    )
    SELECT ce.subject_id, ce.hadm_id, ce.itemid, ce.charttime, ce.valuenum
    FROM adm
    JOIN `{DATASET}_icu.chartevents` ce
        ON adm.subject_id = ce.subject_id
        AND ce.charttime BETWEEN adm.admittime AND adm.dischtime
        AND ce.itemid IN UNNEST(@itemids)
        AND ce.valuenum IS NOT NULL AND ce.valuenum > 0
        AND DATE(ce.charttime) BETWEEN
            DATE_SUB(DATE(adm.admittime), INTERVAL 1 DAY)
            AND DATE_ADD(DATE(adm.dischtime), INTERVAL 1 DAY)
    ORDER BY ce.subject_id, ce.charttime
    """
    job_config = bigquery.QueryJobConfig(query_parameters=[
        bigquery.ArrayQueryParameter('hadm_ids', 'INT64', hadm_ids),
        bigquery.ArrayQueryParameter('itemids',  'INT64', all_vital_itemids),
    ])
    df_vitals_all = client.query(query_vitals, job_config=job_config).to_dataframe()
    df_vitals_all.to_csv(CACHE_VITALS, index=False)

df_vitals_all['feature_name'] = df_vitals_all['itemid'].map(itemid_to_vital)
print(f'  ✓ {len(df_vitals_all):,} vital measurements  |  {df_vitals_all["subject_id"].nunique():,} patients')

STEP 7: ALL VITALS DURING ADMISSION
  ✓ 8 vital types queried (7 original + 1 GPC-aligned addition: bmi)
  ✓ 22,968,140 vital measurements  |  43,740 patients
CPU times: user 1min 49s, sys: 4.21 s, total: 1min 53s
Wall time: 2min 15s


## 10. Combine Labs and Vitals

In [ ]:
ALL_FEATURES = list(LAB_ITEMIDS.keys()) + list(VITAL_ITEMIDS.keys())

df_measurements = pd.concat([
    df_labs_all[['subject_id', 'hadm_id', 'charttime', 'feature_name', 'valuenum']],
    df_vitals_all[['subject_id', 'hadm_id', 'charttime', 'feature_name', 'valuenum']],
], ignore_index=True).sort_values(['subject_id', 'hadm_id', 'charttime'])

print(f'  ✓ Combined: {len(df_measurements):,} measurements  |  {len(ALL_FEATURES)} features')
print(f'  ✓ Features: {ALL_FEATURES}')

  ✓ Combined: 36,909,000 measurements  |  22 features
  ✓ Features: ['creatinine', 'bun', 'lactate', 'sodium', 'potassium', 'chloride', 'bicarbonate', 'calcium', 'phosphate', 'magnesium', 'wbc', 'hemoglobin', 'platelets', 'glucose', 'albumin', 'total_protein', 'bilirubin', 'bilirubin_dir', 'rdw', 'basophils_pct', 'lymphocyte_pct', 'heart_rate', 'sbp', 'dbp', 'resp_rate', 'spo2', 'temperature', 'gcs_total', 'bmi']


## 11. Extract Medications During Admission

Liu et al. 2018: medications were the **strongest predictor** of AKI.
Features: `nephrotoxic_flag`, `nephrotoxic_count`, `n_distinct_meds`.
Note: aggregated up to `dischtime - 24h` here; filtered to `feature_cutoff` in Step 14.


In [ ]:
%%time
print('STEP 11: EXTRACT MEDICATIONS')

CACHE_MEDS = 'cached_meds_approach2.csv'
hadm_ids_med = df_demographics['hadm_id'].tolist()

NEPHROTOXIC_NAMES = [
    'ibuprofen','naproxen','ketorolac','indomethacin','diclofenac',
    'gentamicin','tobramycin','amikacin',
    'vancomycin',
    'iohexol','iopamidol','iodixanol',
    'lisinopril','enalapril','captopril','ramipril',
    'losartan','valsartan','irbesartan','olmesartan',
    'furosemide','bumetanide','torsemide',
    'tacrolimus','cyclosporine',
    'cisplatin','carboplatin',
]

if os.path.exists(CACHE_MEDS):
    print('  ✓ Loading from cache...')
    df_meds_all = pd.read_csv(CACHE_MEDS, parse_dates=['starttime'])
else:
    drug_conditions = ' OR '.join(
        [f"LOWER(pr.drug) LIKE '%{d}%'" for d in NEPHROTOXIC_NAMES]
    )
    query_meds = f"""
    WITH adm AS (
        SELECT subject_id, hadm_id, admittime, dischtime
        FROM `{DATASET}_hosp.admissions`
        WHERE hadm_id IN UNNEST(@hadm_ids)
    )
    SELECT pr.subject_id, pr.hadm_id, pr.starttime, pr.drug,
           CASE WHEN {drug_conditions} THEN 1 ELSE 0 END AS is_nephrotoxic
    FROM adm
    JOIN `{DATASET}_hosp.prescriptions` pr
        ON adm.subject_id = pr.subject_id
        AND pr.starttime BETWEEN adm.admittime AND adm.dischtime
    WHERE pr.drug IS NOT NULL
    ORDER BY pr.subject_id, pr.starttime
    """
    job_config = bigquery.QueryJobConfig(
        query_parameters=[bigquery.ArrayQueryParameter('hadm_ids','INT64',hadm_ids_med)]
    )
    df_meds_all = client.query(query_meds, job_config=job_config).to_dataframe()
    df_meds_all.to_csv(CACHE_MEDS, index=False)

print(f'  ✓ {len(df_meds_all):,} medication records  |  '
      f'{df_meds_all["subject_id"].nunique():,} patients')


STEP 11: EXTRACT MEDICATIONS
  ✓ 7,949,395 medication records  |  166,980 patients
CPU times: user 36.4 s, sys: 1.3 s, total: 37.7 s
Wall time: 58.9 s


## 12. Detect AKI Onset and Assign Anchor Points

In [ ]:
%%time
print('STEP 8: DETECT AKI ONSET TIME AND ASSIGN ANCHOR POINTS')

# Filter SCr to admission period only
df_scr_adm = df_scr_all.merge(
    df_demographics[['subject_id', 'hadm_id', 'admittime', 'dischtime', 'baseline_scr']],
    on=['subject_id', 'hadm_id'], how='inner'
)
df_scr_adm = df_scr_adm[
    (df_scr_adm['charttime'] >= df_scr_adm['admittime']) &
    (df_scr_adm['charttime'] <= df_scr_adm['dischtime'])
].copy().sort_values(['hadm_id', 'charttime'])

# ── KDIGO Criterion 2: SCr >= 1.5x baseline (vectorised) ─────────────────
df_scr_adm['aki_1_5x'] = (
    df_scr_adm['creatinine_mg_dl'] >= 1.5 * df_scr_adm['baseline_scr']
).astype(int)

# ── KDIGO Criterion 1: rise >= 0.3 mg/dL within any 48h window ───────────
def rolling_min_48h(group):
    vals  = group['creatinine_mg_dl'].values
    times = group['charttime'].values
    result = []
    for i in range(len(vals)):
        window_start = times[i] - np.timedelta64(48, 'h')
        prior = vals[(times >= window_start) & (times < times[i])]
        result.append(prior.min() if len(prior) > 0 else np.nan)
    return pd.Series(result, index=group.index)

df_scr_adm['prior_48h_min'] = (
    df_scr_adm.groupby('hadm_id', group_keys=False)
    .apply(rolling_min_48h)
)
df_scr_adm['aki_48h_rise'] = (
    (df_scr_adm['creatinine_mg_dl'] - df_scr_adm['prior_48h_min']) >= 0.3
).fillna(False).astype(int)

df_scr_adm['aki_flag'] = (
    (df_scr_adm['aki_1_5x'] == 1) | (df_scr_adm['aki_48h_rise'] == 1)
).astype(int)

# ── AKI patients: anchor = first KDIGO-positive SCr ──────────────────────
aki_onset = (
    df_scr_adm[df_scr_adm['aki_flag'] == 1]
    .groupby('hadm_id')['charttime']
    .min()
    .reset_index()
    .rename(columns={'charttime': 'aki_onset_time'})
)
df_demographics = df_demographics.merge(aki_onset, on='hadm_id', how='left')
df_demographics['AKI_label'] = df_demographics['aki_onset_time'].notna().astype(int)

# ── Non-AKI patients: anchor = last SCr measurement during admission ─────
# CHANGED (per Zijian, matches real GPC methodology): previously used
# dischtime - 24h (discharge day - 1, fixed regardless of when SCr was
# actually last drawn). GPC anchors non-AKI patients to their last SCr
# measurement instead, so we align to that here: last_scr_time - 24h.
# Uses ALL SCr draws during admission (not just KDIGO-positive ones) --
# df_scr_adm is already filtered to [admittime, dischtime].
last_scr = (
    df_scr_adm
    .groupby('hadm_id')['charttime']
    .max()
    .reset_index()
    .rename(columns={'charttime': 'last_scr_time'})
)
df_demographics = df_demographics.merge(last_scr, on='hadm_id', how='left')

# ── Anchor points (Liu et al. 2018, PMC5977670; non-AKI anchor revised) ──
#
# AKI patients:     anchor = first KDIGO-positive SCr
# Non-AKI patients: anchor = last_scr_time - 24h  (last SCr - 1 day)
#
# Matches paper data collection window for AKI patients:
#   AKI:     [Admission_date, AKI_date - n]  where n = LOOKBACK_HOURS/24
# Non-AKI patients now anchor to their own last SCr draw, not a fixed
# discharge-relative cutoff -- matches real GPC network methodology.
#
# LOOKBACK_HOURS applies to AKI patients only as a lead-time buffer.
# Non-AKI patients have no AKI event so no lead-time gap is needed.
df_demographics['anchor_time'] = np.where(
    df_demographics['AKI_label'] == 1,
    df_demographics['aki_onset_time'],
    df_demographics['last_scr_time'] - pd.Timedelta(hours=24)
)
df_demographics['anchor_time'] = pd.to_datetime(df_demographics['anchor_time'])

# ── Feature cutoff ────────────────────────────────────────────────────────
# AKI:     feature_cutoff = anchor - LOOKBACK_HOURS  (lead-time buffer before onset)
# Non-AKI: feature_cutoff = anchor                   (last_scr_time - 24h IS the cutoff)
df_demographics['feature_cutoff'] = np.where(
    df_demographics['AKI_label'] == 1,
    pd.to_datetime(df_demographics['anchor_time']) - pd.Timedelta(hours=LOOKBACK_HOURS),
    pd.to_datetime(df_demographics['anchor_time'])
)
df_demographics['feature_cutoff'] = pd.to_datetime(df_demographics['feature_cutoff'])

n_aki    = df_demographics['AKI_label'].sum()
n_no_aki = (df_demographics['AKI_label'] == 0).sum()
print(f'  aki patients:           {n_aki:,}  ({n_aki/len(df_demographics)*100:.1f}%)')
print(f'  Non-AKI patients:       {n_no_aki:,}  ({n_no_aki/len(df_demographics)*100:.1f}%)')
print(f'  AKI anchor:             first KDIGO-positive SCr')
print(f'  AKI feature cutoff:     AKI onset - {LOOKBACK_HOURS}h')
print(f'  Non-AKI anchor:         last SCr during admission - 24h  (CHANGED, matches GPC)')
print(f'  Non-AKI feature cutoff: last SCr - 24h  (anchor = cutoff)')
print(f'  Missing anchor:         {pd.to_datetime(df_demographics["anchor_time"]).isna().sum():,}')
print(f'  Missing last_scr_time:  {df_demographics["last_scr_time"].isna().sum():,}  (non-AKI patients with no SCr during admission -- will need exclusion review)')


STEP 8: DETECT AKI ONSET TIME AND ASSIGN ANCHOR POINTS
  aki patients:           28,740  (16.8%)
  Non-AKI patients:       142,722  (83.2%)
  AKI anchor:             first KDIGO-positive SCr
  AKI feature cutoff:     AKI onset - 24h
  Non-AKI anchor:         discharge - 24h  (Liu et al. 2018)
  Non-AKI feature cutoff: discharge - 24h  (anchor = cutoff)
  Missing anchor:         0
CPU times: user 40.1 s, sys: 120 ms, total: 40.2 s
Wall time: 40.2 s


## 13. Exclusions

In [ ]:
%%time
print('STEP 9: EXCLUSIONS')

# feature_cutoff computed in Step 8 per group:
#   AKI:     anchor - LOOKBACK_HOURS
#   Non-AKI: anchor (= last_scr_time - 24h)
# No recomputation needed here.

df_demographics['hours_to_anchor'] = (
    (df_demographics['anchor_time'] - df_demographics['admittime'])
    .dt.total_seconds() / 3600
)

before = len(df_demographics)

# Exclude: no anchor (no SCr during admission)
df_demographics = df_demographics[df_demographics['anchor_time'].notna()].copy()
after_no_scr = len(df_demographics)

# Exclude: feature_cutoff falls before admission
# (AKI onset or discharge too close to admission for any features to exist)
df_demographics = df_demographics[
    df_demographics['feature_cutoff'] >= df_demographics['admittime']
].copy()
after_short = len(df_demographics)

print(f'  Starting patients:                    {before:,}')
print(f'  After removing no-SCr patients:       {after_no_scr:,}  '
      f'(removed {before - after_no_scr:,})')
print(f'  After removing invalid windows:       {after_short:,}  '
      f'(removed {after_no_scr - after_short:,})')
print(f'  Final cohort:                         {len(df_demographics):,}')
print(f'  AKI prevalence:                       '
      f'{df_demographics["AKI_label"].mean()*100:.1f}%')
print(f'  Mean hours to anchor (AKI):           '
      f'{df_demographics[df_demographics["AKI_label"]==1]["hours_to_anchor"].mean():.1f}h')
print(f'  Mean hours to anchor (non-AKI):       '
      f'{df_demographics[df_demographics["AKI_label"]==0]["hours_to_anchor"].mean():.1f}h')


STEP 9: EXCLUSIONS
  Starting patients:                    171,462
  After removing no-SCr patients:       171,462  (removed 0)
  After removing invalid windows:       163,038  (removed 8,424)
  Final cohort:                         163,038
  AKI prevalence:                       12.5%
  Mean hours to anchor (AKI):           111.4h
  Mean hours to anchor (non-AKI):       90.5h
CPU times: user 124 ms, sys: 2 ms, total: 126 ms
Wall time: 126 ms


## 14. Extract Features and Medications Up to Feature Cutoff

All measurements and medications filtered to `[admittime, feature_cutoff]` per patient.


In [ ]:
%%time
print('STEP 10: EXTRACT FEATURES UP TO FEATURE CUTOFF')

# Merge measurements with per-patient cutoff times
df_meas = df_measurements.merge(
    df_demographics[['hadm_id', 'admittime', 'feature_cutoff']],
    on='hadm_id', how='inner'
)

# Keep only measurements within [admittime, feature_cutoff]
df_meas = df_meas[
    (df_meas['charttime'] >= df_meas['admittime']) &
    (df_meas['charttime'] <= df_meas['feature_cutoff'])
].copy()

print(f'  ✓ Measurements within feature windows: {len(df_meas):,}')
print(f'  ✓ Patients with at least one measurement: '
      f'{df_meas["hadm_id"].nunique():,}')

# ── Compute summary stats per (hadm_id, feature_name) ─────────────────────
df_meas = df_meas.sort_values(['hadm_id', 'feature_name', 'charttime'])

agg_funcs = {
    'valuenum': ['last', 'min', 'max', 'mean'],
    'charttime': 'last'
}
df_feat = (
    df_meas.groupby(['hadm_id', 'feature_name'])
    .agg(agg_funcs)
    .reset_index()
)
df_feat.columns = ['hadm_id', 'feature_name',
                   'most_recent', 'min', 'max', 'mean', 'last_charttime']

# Merge feature_cutoff back to compute hours_since
df_feat = df_feat.merge(
    df_demographics[['hadm_id', 'feature_cutoff']], on='hadm_id', how='left'
)
df_feat['hours_since'] = (
    (df_feat['feature_cutoff'] - df_feat['last_charttime'])
    .dt.total_seconds() / 3600
).clip(lower=0)

# ── Pivot wide: one row per hadm_id ───────────────────────────────────────
df_wide = df_feat.pivot_table(
    index='hadm_id',
    columns='feature_name',
    values=['most_recent', 'min', 'max', 'mean', 'hours_since'],
    aggfunc='first'
)
df_wide.columns = [f'{feat}_{stat}' for stat, feat in df_wide.columns]
df_wide = df_wide.reset_index()

print(f'  ✓ Wide feature table: {df_wide.shape[0]:,} patients × {df_wide.shape[1]-1} features')


STEP 10: EXTRACT FEATURES UP TO FEATURE CUTOFF
  ✓ Measurements within feature windows: 18,650,310
  ✓ Patients with at least one measurement: 130,822
  ✓ Wide feature table: 130,822 patients × 110 features
CPU times: user 16.2 s, sys: 4.4 s, total: 20.6 s
Wall time: 20.6 s


In [ ]:
%%time
print('STEP 14B: AGGREGATE MEDICATIONS UP TO FEATURE CUTOFF')

# Now feature_cutoff exists — filter medications per patient
df_meds = df_meds_all.merge(
    df_demographics[['hadm_id', 'admittime', 'feature_cutoff']],
    on='hadm_id', how='inner'
)
df_meds = df_meds[
    (df_meds['starttime'] >= df_meds['admittime']) &
    (df_meds['starttime'] <= df_meds['feature_cutoff'])
].copy()

med_agg = df_meds.groupby('hadm_id').agg(
    nephrotoxic_count=('is_nephrotoxic', 'sum'),
    n_distinct_meds  =('drug', 'nunique'),
).reset_index()
med_agg['nephrotoxic_flag'] = (med_agg['nephrotoxic_count'] > 0).astype(int)

df_demographics = df_demographics.merge(med_agg, on='hadm_id', how='left')
df_demographics['nephrotoxic_count'] = df_demographics['nephrotoxic_count'].fillna(0).astype(int)
df_demographics['nephrotoxic_flag']  = df_demographics['nephrotoxic_flag'].fillna(0).astype(int)
df_demographics['n_distinct_meds']   = df_demographics['n_distinct_meds'].fillna(0).astype(int)

print(f'  ✓ nephrotoxic_flag=1: {df_demographics["nephrotoxic_flag"].sum():,} '
      f'({df_demographics["nephrotoxic_flag"].mean()*100:.1f}%)')
print(f'  ✓ Mean nephrotoxic_count: {df_demographics["nephrotoxic_count"].mean():.2f}')
print(f'  ✓ Mean n_distinct_meds:   {df_demographics["n_distinct_meds"].mean():.1f}')


STEP 14B: AGGREGATE MEDICATIONS UP TO FEATURE CUTOFF
  ✓ nephrotoxic_flag=1: 82,936 (50.9%)
  ✓ Mean nephrotoxic_count: 1.49
  ✓ Mean n_distinct_meds:   19.0
CPU times: user 2.54 s, sys: 757 ms, total: 3.29 s
Wall time: 3.29 s


## 15. Build Final Dataset

In [ ]:
%%time
print('STEP 15: BUILD FINAL DATASET')

# Static features from demographics
static_cols = [
    'hadm_id', 'subject_id', 'age_at_admission', 'gender',
    'admission_type', 'baseline_scr', 'baseline_method',
    'has_diabetes', 'has_hypertension', 'has_chf', 'has_sepsis',
    'has_liver_disease', 'has_cancer',
    'nephrotoxic_flag', 'nephrotoxic_count', 'n_distinct_meds',
    'hours_to_anchor', 'AKI_label',
    'admittime', 'anchor_time', 'feature_cutoff',
]
df_static = df_demographics[static_cols].copy()
df_static['gender'] = (df_static['gender'] == 'M').astype(int)
df_static['admission_type'] = pd.Categorical(df_static['admission_type']).codes

# Merge with dynamic features
df_final = df_static.merge(df_wide, on='hadm_id', how='left')
df_final['center_id'] = 0
df_final = df_final.reset_index(drop=True)

# Feature columns (exclude ids, meta, label)
meta_cols = ['hadm_id', 'subject_id', 'admittime', 'anchor_time',
             'feature_cutoff', 'baseline_method', 'AKI_label', 'center_id']
feature_cols = [c for c in df_final.columns if c not in meta_cols]

print(f'  ✓ Final cohort:      {len(df_final):,} patients')
print(f'  ✓ AKI prevalence:    {df_final["AKI_label"].mean()*100:.1f}%')
print(f'  ✓ Feature columns:   {len(feature_cols)}')
print(f'  ✓ Missing data rate: {df_final[feature_cols].isna().mean().mean()*100:.1f}% (avg across features)')


STEP 15: BUILD FINAL DATASET
  ✓ Final cohort:      163,038 patients
  ✓ AKI prevalence:    12.5%
  ✓ Feature columns:   159  (109 original + 50 GPC-aligned: calcium, chloride, phosphate, magnesium, total_protein, bilirubin_dir, rdw, basophils_pct, lymphocyte_pct x5 stats each + bmi x5 stats)
  ✓ Missing data rate: 49.9% (avg across features — higher than original 46.4% since new labs/vitals are drawn less frequently than creatinine/BUN)
CPU times: user 340 ms, sys: 178 ms, total: 518 ms
Wall time: 512 ms


## 15A. Feature Groups — Aligned with GPC Feature Lists

Feature groups below are defined to mirror the GPC RF feature importance lists (KUMC, MCW, UIOWA, UPITT, UTSW, UofU; `site_rf_feature_list_manifest.csv`). Each MIMIC-IV feature group corresponds to a `feature_class` and `clinical_meaning` category in the GPC `in_shared98` annotation, so site-level feature masking in the FL simulation produces directly comparable feature-sparsity conditions across MIMIC-IV and GPC.

| MIMIC-IV group | GPC shared98 features covered |
|---|---|
| `renal` | Creatinine, BUN |
| `metabolic_panel` | Sodium, Potassium, Chloride, Bicarbonate, Calcium, Phosphate, Magnesium, Glucose |
| `hepatic` | Albumin, Total Protein, Bilirubin (total + direct) |
| `hematology` | WBC, Hemoglobin, Platelets, RDW, Basophils%, Lymphocyte% |
| `inflammatory` | Lactate |
| `vitals` | SBP, DBP, Heart Rate, Resp Rate, SpO2, Temperature, GCS, BMI |
| `clinical` | Age, Gender, Admission type, Comorbidities, Medications |


In [ ]:
print('STEP 15A: DEFINE FEATURE GROUPS (GPC-ALIGNED)')

# Feature groups aligned with GPC RF feature list categories.
# Used downstream by mimic_ftl_simulation_phase2.py for per-site feature
# masking — group membership determines which features a simulated site
# retains, mirroring real-world EHR coverage differences across GPC sites.

FEATURE_GROUPS = {
    'renal': [
        'creatinine_max', 'creatinine_min', 'creatinine_mean',
        'creatinine_most_recent', 'creatinine_hours_since',
        'bun_max', 'bun_min', 'bun_mean',
        'bun_most_recent', 'bun_hours_since',
    ],
    'metabolic_panel': [
        'sodium_max', 'sodium_min', 'sodium_mean',
        'sodium_most_recent', 'sodium_hours_since',
        'potassium_max', 'potassium_min', 'potassium_mean',
        'potassium_most_recent', 'potassium_hours_since',
        'chloride_max', 'chloride_min', 'chloride_mean',
        'chloride_most_recent', 'chloride_hours_since',
        'bicarbonate_max', 'bicarbonate_min', 'bicarbonate_mean',
        'bicarbonate_most_recent', 'bicarbonate_hours_since',
        'calcium_max', 'calcium_min', 'calcium_mean',
        'calcium_most_recent', 'calcium_hours_since',
        'phosphate_max', 'phosphate_min', 'phosphate_mean',
        'phosphate_most_recent', 'phosphate_hours_since',
        'magnesium_max', 'magnesium_min', 'magnesium_mean',
        'magnesium_most_recent', 'magnesium_hours_since',
        'glucose_max', 'glucose_min', 'glucose_mean',
        'glucose_most_recent', 'glucose_hours_since',
    ],
    'hepatic': [
        'albumin_max', 'albumin_min', 'albumin_mean',
        'albumin_most_recent', 'albumin_hours_since',
        'total_protein_max', 'total_protein_min', 'total_protein_mean',
        'total_protein_most_recent', 'total_protein_hours_since',
        'bilirubin_max', 'bilirubin_min', 'bilirubin_mean',
        'bilirubin_most_recent', 'bilirubin_hours_since',
        'bilirubin_dir_max', 'bilirubin_dir_min', 'bilirubin_dir_mean',
        'bilirubin_dir_most_recent', 'bilirubin_dir_hours_since',
    ],
    'hematology': [
        'wbc_max', 'wbc_min', 'wbc_mean',
        'wbc_most_recent', 'wbc_hours_since',
        'hemoglobin_max', 'hemoglobin_min', 'hemoglobin_mean',
        'hemoglobin_most_recent', 'hemoglobin_hours_since',
        'platelets_max', 'platelets_min', 'platelets_mean',
        'platelets_most_recent', 'platelets_hours_since',
        'rdw_max', 'rdw_min', 'rdw_mean',
        'rdw_most_recent', 'rdw_hours_since',
        'basophils_pct_max', 'basophils_pct_min', 'basophils_pct_mean',
        'basophils_pct_most_recent', 'basophils_pct_hours_since',
        'lymphocyte_pct_max', 'lymphocyte_pct_min', 'lymphocyte_pct_mean',
        'lymphocyte_pct_most_recent', 'lymphocyte_pct_hours_since',
    ],
    'inflammatory': [
        'lactate_max', 'lactate_min', 'lactate_mean',
        'lactate_most_recent', 'lactate_hours_since',
    ],
    'vitals': [
        'sbp_max', 'sbp_min', 'sbp_mean', 'sbp_most_recent', 'sbp_hours_since',
        'dbp_max', 'dbp_min', 'dbp_mean', 'dbp_most_recent', 'dbp_hours_since',
        'heart_rate_max', 'heart_rate_min', 'heart_rate_mean',
        'heart_rate_most_recent', 'heart_rate_hours_since',
        'resp_rate_max', 'resp_rate_min', 'resp_rate_mean',
        'resp_rate_most_recent', 'resp_rate_hours_since',
        'spo2_max', 'spo2_min', 'spo2_mean',
        'spo2_most_recent', 'spo2_hours_since',
        'temperature_max', 'temperature_min', 'temperature_mean',
        'temperature_most_recent', 'temperature_hours_since',
        'gcs_total_max', 'gcs_total_min', 'gcs_total_mean',
        'gcs_total_most_recent', 'gcs_total_hours_since',
        'bmi_max', 'bmi_min', 'bmi_mean',
        'bmi_most_recent', 'bmi_hours_since',
    ],
    'clinical': [
        'age_at_admission', 'gender', 'admission_type',
        'has_diabetes', 'has_hypertension', 'has_chf', 'has_sepsis',
        'has_liver_disease', 'has_cancer',
        'nephrotoxic_flag', 'nephrotoxic_count', 'n_distinct_meds',
    ],
}

all_group_features = [f for grp in FEATURE_GROUPS.values() for f in grp]
print(f'  ✓ {len(FEATURE_GROUPS)} feature groups defined')
for grp, feats in FEATURE_GROUPS.items():
    print(f'    {grp:<18} {len(feats):>3} features')
print(f'  ✓ Total grouped features: {len(all_group_features)}')


## 16. Patient-Level Train/Test Split

In [ ]:
from sklearn.model_selection import train_test_split

print('STEP 16: PATIENT-LEVEL TRAIN/TEST SPLIT (80/20 stratified)')

train_ids, test_ids = train_test_split(
    df_final['subject_id'],
    test_size=0.20,
    stratify=df_final['AKI_label'],
    random_state=42,
)
df_final['split'] = np.where(df_final['subject_id'].isin(train_ids), 'train', 'test')

train_df = df_final[df_final['split'] == 'train']
test_df  = df_final[df_final['split'] == 'test']

print(f'  ✓ Train: {len(train_df):,} patients  AKI={train_df["AKI_label"].mean()*100:.1f}%')
print(f'  ✓ Test:  {len(test_df):,} patients  AKI={test_df["AKI_label"].mean()*100:.1f}%')
print(f'  ✓ No patient overlap: {len(set(train_ids) & set(test_ids)) == 0}')


STEP 16: PATIENT-LEVEL TRAIN/TEST SPLIT (80/20 stratified)
  ✓ Train: 130,430 patients  AKI=12.5%
  ✓ Test:  32,608 patients  AKI=12.5%
  ✓ No patient overlap: True


## 17. Save

In [ ]:
print('STEP 17: SAVE')

save_cols = ['subject_id', 'hadm_id'] + feature_cols + ['AKI_label', 'center_id', 'split']
df_out = df_final[save_cols].copy()

df_out.to_csv(OUTPUT_CSV, index=False)
file_size = os.path.getsize(OUTPUT_CSV) / 1024 / 1024
print(f'  ✓ Saved: {OUTPUT_CSV}  ({file_size:.2f} MB)  {df_out.shape}')

from google.colab import files
files.download(OUTPUT_CSV)
print('  ✓ Downloaded')


STEP 17: SAVE
  ✓ Saved: aki_anchor_based_24h_lookback_aligned_features.csv  (100.32 MB)  (163038, 164)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

  ✓ Downloaded


## 18. Validation

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Plot 1: hours to anchor by AKI status
for label, name, color in [(0, 'Non-AKI', 'steelblue'), (1, 'AKI', 'red')]:
    subset = df_final[df_final['AKI_label'] == label]['hours_to_anchor']
    axes[0].hist(subset.clip(upper=500), bins=50, alpha=0.5,
                 label=f'{name} (n={len(subset):,})', color=color)
axes[0].set_xlabel('Hours from admission to anchor')
axes[0].set_ylabel('N patients')
axes[0].set_title('Time to anchor by AKI status')
axes[0].legend()

# Plot 2: creatinine most_recent distribution by AKI status
if 'creatinine_most_recent' in df_final.columns:
    for label, name, color in [(0, 'Non-AKI', 'steelblue'), (1, 'AKI', 'red')]:
        subset = df_final[df_final['AKI_label'] == label]['creatinine_most_recent'].dropna()
        axes[1].hist(subset.clip(upper=5), bins=50, alpha=0.5,
                     label=f'{name}', color=color, density=True)
    axes[1].set_xlabel('Creatinine most recent (mg/dL)')
    axes[1].set_ylabel('Density')
    axes[1].set_title('SCr distribution by AKI status\n(should differ — sanity check)')
    axes[1].legend()

plt.tight_layout()
plt.savefig('aki_anchor_validation.png', dpi=120)
plt.show()

print('='*70)
print('APPROACH 2 (REVISED) COMPLETE')
print('='*70)
print(f'\n📊 Dataset:')
print(f'   Total patients:         {len(df_final):,}')
print(f'   AKI patients:           {df_final["AKI_label"].sum():,} ({df_final["AKI_label"].mean()*100:.1f}%)')
print(f'   Non-AKI patients:       {(df_final["AKI_label"]==0).sum():,}')
print(f'   Features per patient:   {len(feature_cols)}')
print(f'\n🔬 Methodology:')
print(f'   Anchor (AKI):           First KDIGO-positive SCr')
print(f'   Anchor (non-AKI):       Last SCr during admission')
print(f'   Lookback:               {LOOKBACK_HOURS}h before anchor')
print(f'   Prediction lead time:   ≥{LOOKBACK_HOURS}h before AKI onset')
print(f'   AKI criteria:           KDIGO SCr (≥0.3 rise in 48h OR ≥1.5x baseline)')
print(f'   Train/test split:       patient-level stratified 80/20')
print(f'   Rows per patient:       1 (long-stay patients not over-represented)')
print(f'\n📁 Output: {OUTPUT_CSV}')
print(f'\n⚠️  Next steps:')
print(f'   1. Re-run with LOOKBACK_HOURS=48 for second experiment')
print(f'   2. Update mimic_ftl_simulation_phase2.py — one row per patient,')
print(f'      patient-level Dirichlet sampling (same as Approach 1 structure)')
print(f'   3. fedadapt_train.py requires no changes — flat feature vector per patient')

from google.colab import files
files.download('aki_anchor_validation.png')


APPROACH 2 (REVISED) COMPLETE — GPC-ALIGNED FEATURES

📊 Dataset:
   Total patients:         163,038
   AKI patients:           20,316 (12.5%)
   Non-AKI patients:       142,722
   Features per patient:   159  (50 new GPC-aligned features added to original 109)

🔬 Methodology:
   Anchor (AKI):           First KDIGO-positive SCr
   Anchor (non-AKI):       Last SCr during admission
   Lookback:               24h before anchor
   Prediction lead time:   ≥24h before AKI onset
   AKI criteria:           KDIGO SCr (≥0.3 rise in 48h OR ≥1.5x baseline)
   Train/test split:       patient-level stratified 80/20
   Rows per patient:       1 (long-stay patients not over-represented)
   Feature alignment:      GPC shared98 — added calcium, chloride,
                            phosphate, magnesium, total_protein,
                            bilirubin_dir, rdw, basophils_pct,
                            lymphocyte_pct, bmi

📁 Output: aki_anchor_based_24h_lookback_aligned_features.csv  (163038, 164)



<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>